In [1]:
import os


In [2]:
%pwd

'c:\\Users\\Hp\\Desktop\\End-to-End-Machine_learning_project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Hp\\Desktop\\End-to-End-Machine_learning_project'

entity

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict # save parameter 
    metric_file_name : Path
    target_column : str




update the configuration manger is src config

In [6]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self)-> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])
        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path= config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name

        )


        return model_evaluation_config


component

In [8]:
import pandas as pd
import os
import logging
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
from mlProject.utils.common import save_json
import numpy as np
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)

        return rmse, mae, r2


    def save_result(self):
        test_data  = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)


        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        predicted_qualities = model.predict(test_x)

        (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

        #saving metrics as local
        scores = {"rmse": rmse, "mae":mae, "r2":r2}
        save_json(path=Path(self.config.metric_file_name), data=scores)



    


In [10]:
try:
    config = ConfigurationManager()
    
    model_eval_config = config.get_model_evaluation_config()
    
    model_evaluation = ModelEvaluation(config=model_eval_config)
    
    model_evaluation.save_result()

except Exception as e:
    raise e

[2026-04-25 09:45:24,913: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-25 09:45:24,917: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-25 09:45:24,924: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-04-25 09:45:24,926: INFO: common: created directory at: artifacts]
[2026-04-25 09:45:24,927: INFO: common: created directory at: artifacts/model_trainer]


EnsureError: Argument path of type <class 'str'> to <function save_json at 0x000001B69E8BEDC0> does not match annotation type <class 'pathlib.Path'>